In [ ]:
# ===== PPO ARM: CONTROL (unshaped baseline reward) — CONFIG (edit ONLY this cell) =====
# Runtime: A100. This notebook is ONE ARM of a two-arm experiment. Run it and its
# sibling `ppo_arm_s20mk2.ipynb` side by side on two A100s: they are generated from
# one template and differ in exactly four lines -- ARM, SHAPING, LAMBDA, and the Drive
# folder -- so any difference in their results is the reward and nothing else.
#
# The reward the 610model was trained with, unchanged:
#   reward = 1000 if terminated else -min(nnz, 10)
# This arm exists because our run differs from the published one in several
# ways at once -- held-out data, PyTorch instead of JAX, possibly TF32 -- so
# the shaped arm cannot be read against the 610model's 49/60. It can only be
# read against THIS.

ARM      = "control"
SHAPING  = None       # None = control; "s20mk2" = L + 20*S + 2*MK
LAMBDA   = 0.0           # strength of the shaping term; 0 is a hard zero

REPO_URL = "https://github.com/Avi161/ACSolverX.git"
BRANCH   = "experiments/ppo"
REPO_DIR = "ACSolverX"
CLONE       = True
UPDATE_REPO = True            # git reset --hard so a RESTART pulls the latest push
MOUNT_DRIVE = True
INSTALL_REAL_DISTRAX = False  # run_ppo ships a shim verified bit-identical to distrax

# --- the training set -------------------------------------------------------
# `AC19_extended` with the 60 benchmark presentations removed. It is BUILT on the VM
# from the shipped file plus the frozen benchmark CSV -- not committed -- so it cannot
# drift from the benchmark it is defined against.
#
# Why it matters: 54 of the 60 evaluation presentations are in `AC19_extended`, and all
# 54 sit inside the first 634 lines, which is the block that gets PINNED one-env-each for
# the whole run. Training on the shipped file means training on 54/60 of the test set.
#
# The removal has a trap, already handled in the runner: benchmark row `bin 0` is line 0
# of the file, so the shipped prefix walk (`acs_data.ms_prefix_length`) returns 0 on the
# filtered file and pinning would switch OFF entirely, in both arms, silently.
# `heldout.ms_prefix_length` counts membership instead and gives 580 = 634 - 54.
DATASET = "AC19_extended_ho60"
SEED    = 142

# 1000 updates, the step count of the shipped 610model. Measured at 31.2 s/update with
# TF32 off, so ~8.7 h. Resumable: a disconnect continues from the checkpoint, and raising
# this later costs only the extra updates.
MAX_UPDATES = 1000

# --- smoke ------------------------------------------------------------------
# True -> 2 updates and a time-bounded benchmark-60 eval, ~8 minutes. Run it ONCE per
# arm before the real thing. It is not a formality: it measures seconds/update on THIS
# machine and proves the optimiser step, the checkpoint, the Drive mirror and the
# evaluation all fire. Compare its seconds_per_update against 31.18 (TF32 off) to see
# what ALLOW_TF32 bought.
SMOKE_RUN     = True
SMOKE_SECONDS = 300

# --- output -----------------------------------------------------------------
# SEPARATE Drive folders per arm. The checkpoints and jsonl already carry arm-specific
# tags, but `smoke_report.json` and `report_history.jsonl` are fixed names -- two live
# sessions mirroring to one folder would race on them.
LOCAL_OUT_DIR = "results/ppo/control"
DRIVE_OUT_DIR = "/content/drive/MyDrive/acsolverx_results/ppo/control"

cfg = {
    "DEVICE": "auto",

    # 74% of an update is the optimiser, not rollout collection (collect 8.0 s vs learn
    # 23.2 s measured), and the batch is 228,480 transitions through a 137k-parameter
    # net -- large-batch fp32 matmul, which is exactly what TF32 accelerates. The parity
    # stage forces TF32 OFF regardless, so the gate is unaffected, and both arms use the
    # same setting, so the comparison is unaffected too.
    "ALLOW_TF32": True,

    "MICRO_BATCH": 2048,
    "ROLLOUT_CHUNK": 4096,
    "SAVE_EVERY": 25,
    "HEARTBEAT_EVERY_S": 60,

    "CKPT_DIR":   "ppo_checkpoints/610model",
    "CKPT_STEP":  None,
    "PARAMS_NPZ": "ppo_checkpoints/610model_params.npz",

    # --- evaluation ---------------------------------------------------------
    # benchmark-60, by standing rule. 60 rows are only 45 Aut orbits, so every headline
    # is reported both ways. Bins 0-6 saturate for any competent model (a TWO-UPDATE
    # model scored 318/331 on the easy head where the trained one scored 331/331), so
    # the arms can only differ on bins 7-9: 18 rows, 11 orbits. Thin, and said up front.
    "EVAL_DATASET": "benchmark60",
    "EVAL_START": 0,
    "EVAL_END": None,
    "BEAM_WIDTH": 1024,
    "BEAM_MAX_STEPS": 150,
    "BEAM_ALPHA": 0.0,
    "BEAM_TEMPERATURE": 0.0,
    "BEAM_TEMP_END": 0.0,
    "BEAM_TIME_BUDGET_S": None,
    "PARITY_BATCH": 256,

    "USE_WANDB": True,
    "AUTO_AUTHENTICATE_WANDB": False,
    "WANDB_ENTITY": "avigyapaudel045-aisc",
    "WANDB_PROJECT": "acsolver",
    "WANDB_GROUP": "ppo-shaped-reward",     # one group, both arms, so they overlay
    "WANDB_JOB_TYPE": "ppo-arm-control",
    "WANDB_TAGS": ["ppo", "shaped-reward", "control"],
    "WANDB_NOTES": None,
}

# --- swap-ins ---------------------------------------------------------------
# The full run, after the smoke:      SMOKE_RUN = False
# A second seed once one is done:     SEED = 1        (new tag, new checkpoint)
# Gentler shaping if training is unstable (treatment arm only):  LAMBDA = 0.1

print(f"config loaded: arm={ARM} shaping={SHAPING} lambda={LAMBDA}")


In [ ]:
# ==================== SETUP (clone / pull / install / mount / auth) ========
import os, sys, subprocess

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    BASE = "/content"
    os.chdir(BASE)                       # anchor so re-runs never nest the clone
    if not os.path.isdir(REPO_DIR):
        if CLONE:
            sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    # torch and jax are PREINSTALLED on the Colab GPU image with matching CUDA
    # wheels -- reinstalling either is how a working runtime gets broken. Only
    # what is missing: flax (the JAX net the parity gate compares against),
    # orbax-checkpoint (reads ppo_checkpoints/610model), and wandb.
    sh("pip -q install flax orbax-checkpoint wandb")
    # distrax: OFF by default, and on its own line so a failure here stops
    # nothing. See INSTALL_REAL_DISTRAX in CONFIG for why the shim is preferred.
    if INSTALL_REAL_DISTRAX:
        sh("pip -q install distrax")
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    REPO_ROOT = os.path.join(BASE, REPO_DIR)
else:
    # local: walk up from cwd to the repo root (dir holding experiments/ + data/)
    REPO_ROOT = os.getcwd()
    while REPO_ROOT != "/" and not (
        os.path.isdir(os.path.join(REPO_ROOT, "experiments"))
        and os.path.isdir(os.path.join(REPO_ROOT, "data"))
    ):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

# run from repo root so "data/..." and "import experiments..." resolve
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)

# The production beam budget (1024 x 150 = 153,600 expansions) is far above the
# repo's local cap; the cap exists so a laptop session cannot start a production
# search by accident. This notebook IS the production run, so it opts in.
os.environ["ACSOLVERX_ALLOW_BIG"] = "1"

# SETUP's `git reset --hard` rewrites the .py files on disk, but Python keeps the
# OLD module objects in sys.modules for the life of the runtime -- so the RUN
# cell's `from experiments.ppo.run_ppo import main` would silently reuse stale
# code (a pull is NOT a reload). Drop them so the next import reads what SETUP
# just fetched. Without this you must Runtime -> Restart session.
import importlib
for _m in [m for m in sys.modules if m == "experiments" or m.startswith("experiments.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

# ---- the runtime we actually got ----------------------------------------
if IN_COLAB:
    sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")
import torch
print(f"torch {torch.__version__}  cuda={torch.cuda.is_available()} "
      f"device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
if not torch.cuda.is_available():
    print("WARNING: no GPU. `train` on CPU is not viable; Runtime -> Change runtime type -> A100.")

# ---- W&B authentication -------------------------------------------------
# AUTO_AUTHENTICATE_WANDB (set in CONFIG):
#   True  -> PROMPTLESS auth from a Colab Secret named WANDB_API_KEY (persists
#            across runtime restarts). Create it ONCE: Colab left sidebar ->
#            key icon (Secrets) -> add WANDB_API_KEY = <your key> -> toggle
#            'Notebook access' ON. Without a secret it reuses a valid key from
#            this session, else prompts once (a stale/invalid key is discarded,
#            not reused -- so it can always recover).
#   False -> prompt for the API key EVERY run (use to switch/refresh a key).
if cfg["USE_WANDB"]:
    import re, wandb

    def _clean_key(k):
        # strip whitespace/newlines and stray surrounding quotes from a paste
        return (k or "").strip().strip('"').strip("'").strip()

    def _valid(k):
        return bool(re.fullmatch(r"[A-Za-z0-9_]+", k or ""))

    def _prompt_key():
        import getpass
        return _clean_key(getpass.getpass(
            "Paste your W&B API key (from wandb.ai/authorize), then Enter: "))

    _auto = cfg.get("AUTO_AUTHENTICATE_WANDB", True)
    if not _auto:
        os.environ["WANDB_API_KEY"] = _prompt_key()          # always ask for a fresh key
    else:
        # auto: prefer a Colab Secret; else reuse a VALID session key; else prompt.
        _key, _why = None, ""
        try:
            from google.colab import userdata
            _key = _clean_key(userdata.get("WANDB_API_KEY"))
        except Exception as _e:
            _why = type(_e).__name__   # SecretNotFoundError / NotebookAccessError
        if _key:
            os.environ["WANDB_API_KEY"] = _key
            print("W&B: using Colab Secret WANDB_API_KEY (promptless).")
        elif _valid(os.environ.get("WANDB_API_KEY", "")):
            print("W&B: reusing the key from this session.")
        else:
            if os.environ.get("WANDB_API_KEY"):     # stale/invalid -> drop it, don't reuse
                print("W&B: discarding an invalid key left in this session.")
                os.environ.pop("WANDB_API_KEY", None)
            print("W&B: no usable Colab Secret WANDB_API_KEY"
                  f"{(' (' + _why + ')') if _why else ''}.")
            print("     -> Promptless auth: Colab left sidebar -> key icon (Secrets)")
            print("        -> Add  name=WANDB_API_KEY  value=<key from wandb.ai/authorize>")
            print("        -> toggle 'Notebook access' ON, then re-run this cell.")
            print("     Pasting once for THIS session instead:")
            os.environ["WANDB_API_KEY"] = _prompt_key()

    # final format check before hitting the server (Colab getpass can mangle a paste)
    if not _valid(os.environ.get("WANDB_API_KEY", "")):
        print("W&B: that key still has invalid characters (allowed: A-Z a-z 0-9 _).")
        print("     Re-copy it EXACTLY from https://wandb.ai/authorize.")

    # verify; relogin=True overwrites any stale key cached in ~/.netrc
    try:
        wandb.login(key=os.environ.get("WANDB_API_KEY", ""), relogin=True, verify=True)
        try:
            _default = wandb.Api().default_entity
        except Exception:
            _default = None
        _target = cfg["WANDB_ENTITY"] or _default
        print(f"W&B: authenticated ✓  runs -> {_target}/{cfg['WANDB_PROJECT']}")
    except Exception as e:
        print(f"W&B: authentication FAILED ✗ -- {e}")
        print("     tip: add a Colab Secret WANDB_API_KEY (promptless), or set "
              "AUTO_AUTHENTICATE_WANDB=False and re-run to paste a fresh key.")

In [ ]:
# ==================== RUN =================================================
# Restart -> Run All continues: `train` resumes from its checkpoint and `beam_eval`
# skips presentations already in its jsonl. Nothing here needs a manual step -- the
# held-out training file and the 60-row evaluation file are built on first use.
import os
from experiments.ppo import bench60
from experiments.ppo.ppo import make_config
from experiments.ppo.run_ppo import main, train_tag

cfg["OUT_DIR"] = os.path.join(REPO_ROOT, LOCAL_OUT_DIR)
cfg["MIRROR_DIR"] = DRIVE_OUT_DIR if (IN_COLAB and MOUNT_DRIVE) else None
cfg["DATASET"] = DATASET
cfg["SEED"] = SEED
cfg["SHAPING"] = SHAPING
cfg["LAMBDA"] = LAMBDA
cfg["MAX_UPDATES"] = MAX_UPDATES

# `parity` is kept in the ladder even though this arm trains from scratch: it carries
# the env self-check, and the reward path is the thing under test.
STAGES = ["convert", "parity", "train", "bench60", "report"]
if SMOKE_RUN:
    cfg["MAX_UPDATES"] = 2          # meaningless numbers, real code path
    cfg["SAVE_EVERY"] = 1           # so the checkpoint and the mirror are exercised
    cfg["BEAM_TIME_BUDGET_S"] = SMOKE_SECONDS
    cfg["USE_WANDB"] = False

TAG = train_tag(DATASET, SEED, SHAPING, LAMBDA)
print(f"{'SMOKE' if SMOKE_RUN else 'FULL'} RUN: {' -> '.join(STAGES)}")
print(f"  arm={ARM}  tag={TAG}  updates={cfg['MAX_UPDATES']}  tf32={cfg['ALLOW_TF32']}")

BASE = dict(make_config())
BASE.update(cfg)

def run(stage, **over):
    c = dict(BASE)
    c.update(over)
    c["STAGE"] = stage
    return main(c)

results = {}

if "convert" in STAGES:
    results["convert"] = run("convert")

if "parity" in STAGES:
    results["parity"] = run("parity")

if "train" in STAGES:
    results["train"] = run("train")

if "bench60" in STAGES:
    # An ordinary beam over a 60-row dataset, so it resumes, verifies every certificate
    # and reports through the same code as the 1190-row run. Re-runnable at any time
    # against the checkpoint on disk -- including from a second session, mid-training.
    ckpt = os.path.join(cfg["OUT_DIR"], TAG + ".pt")
    if os.path.exists(ckpt):
        results["bench60"] = run("beam_eval", BEAM_CHECKPOINT=ckpt)
    else:
        print(f"skip bench60: no checkpoint at {ckpt} yet")

if "report" in STAGES:
    results["report"] = run("report", SMOKE_RUN=SMOKE_RUN)

print("\ndone:", ", ".join(results))


In [ ]:
# ==================== TABLE ===============================================
# The deliverable. A different lifetime from the run: it only reads disk, so
# re-printing it never re-runs a stage, and it can be run against a partially
# trained checkpoint at any point.
#
# Reported at ROW level and ORBIT level, always both -- benchmark-60's 60 rows are
# 45 Aut orbits (class 106 alone appears 8 times), so a bare n/60 overstates any
# method that happens to suit a duplicated orbit. `bins7-9` is the subset the arms
# can actually differ on; bins 0-6 saturate.
from experiments.ppo import bench60

print(bench60.format_table(bench60.summarise(cfg["OUT_DIR"])))
